# cdm_skani_gtdb CTS Demo

End-to-end demo of `cdm_skani_gtdb`: nearest-reference ANI for user genomes against the
**GTDB R232** representative sketch panel. Returns the same kind of `closest_genome_*` info that
`gtdbtk classify_wf` produces, but in seconds instead of hours, and with top-N hits per query
instead of the collapsed top-1 that gtdbtk emits.

- **Image:** `ghcr.io/kbaseincubator/cdm_skani_gtdb:0.1.0@sha256:682e6d44512cb8911d05459d00dd947ad7d71ebb28c380aaaae6a2a5efe856c5`
- **Refdata UUID:** `bb6352b4-b86f-4e3d-a858-4bc77327ab13` (same R232 bundle that cdm_gtdbtk uses)
- **Refdata mount:** `/ref_data/` (skani sketch DB at `/ref_data/release232/skani/database/`)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/skani_gtdb_demo/`

See the [cdm_skani_gtdb repo](https://github.com/kbaseincubator/cdm_skani_gtdb) for full details
including why we pin to R232 (rather than skani's older pre-sketched R226) and how this pairs
with cdm_gtdbtk.

**What we'll show:**
1. Submit one `skani search` job: all 4 test genomes as queries, top-5 nearest GTDB reps each.
2. Parse the hits TSV into a per-query top-N view.
3. Cross-check the top-1 hit against `gtdbtk.{bac120,ar53}.summary.tsv:closest_genome_reference`
   from the gtdbtk demo run, to verify that two independent calls into the same R232 bundle
   agree on the closest reference per genome.


## 1. Setup

In [1]:
import io, time
import pandas as pd

tscli  = get_task_service_client()
mincli = get_minio_client()

IMAGE      = "ghcr.io/kbaseincubator/cdm_skani_gtdb:0.1.0@sha256:682e6d44512cb8911d05459d00dd947ad7d71ebb28c380aaaae6a2a5efe856c5"
OUTPUT_DIR = "cts/io/jplfaria/output/skani_gtdb_demo"
SKETCH_DB  = "/ref_data/release232/skani/database/"

# Cached gtdbtk demo output we'll cross-check against
GTDBTK_DEMO_PREFIX = "io/jplfaria/output/gtdbtk/test/v1/"

print(tscli.whoami())


{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Same 4 test assemblies the gtdbtk demo uses, so the cross-check below is apples-to-apples.


In [2]:
input_files = sorted(
    f"cts/{o.object_name}"
    for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True)
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna")
)
print(f"{len(input_files)} query genome(s):")
for f in input_files:
    print(f"  {f}")


4 query genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. Submit skani search against R232 sketch DB

One container, all 4 inputs as queries. `-n 5` asks for top-5 nearest GTDB representatives per
query; `--short-header` strips long contig descriptions. Memory budget is generous (32 GB);
skani search itself fits in roughly 6 GB even against the full ~140k-genome sketch.


In [3]:
job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "search",
        "-d", SKETCH_DB,
        "-o", "/out/hits.tsv",
        "-t", "4",
        "-n", "5",
        "--short-header",
        tscli.insert_files(),
    ],
    num_containers=1,
    cpus=4, memory="32GB", runtime="PT30M",
)
print(f"job: {job.id}")
t0 = time.time()
job.wait_for_completion()
print(f"  state={job.get_job_status()['state']}  exit={job.get_exit_codes().get('exit_codes')}  waited={time.time()-t0:.0f}s")


job: 4b7fa2b0-b131-41f6-8257-8ed0d6ad66a2


  state=complete  exit=[0]  waited=40s


## 4. Read and inspect the hits TSV

Columns: `Ref_file`, `Query_file`, `ANI`, `Align_fraction_ref`, `Align_fraction_query`,
`Ref_name`, `Query_name`. The `Ref_file` path is the original GTDB build-time absolute path
(stored at sketch time, not the runtime mount path), but the GCA / GCF accession at the end of
the path is the GTDB representative ID we want.


In [4]:
outs = [o for o in job.get_job().get("outputs", []) if o["file"].endswith("hits.tsv")]
assert outs, "no hits.tsv in outputs"
bucket, key = outs[0]["file"].split("/", 1)
raw = mincli.get_object(bucket, key).read().decode("utf-8")
print(f"hits.tsv: {len(raw)} bytes")

hits = pd.read_csv(io.StringIO(raw), sep="\t")
print(f"\nrows: {len(hits)}, queries: {hits['Query_file'].nunique()}")

# Pull the GCA/GCF accession from each Ref_file basename (it ends with e.g. GCA_000147015.1_genomic.fna.gz)
import re
def gtdb_acc(p):
    m = re.search(r"(GC[AF]_\d+\.\d+)", str(p))
    return m.group(1) if m else None

def query_acc(p):
    m = re.search(r"(GC[AF]_\d+\.\d+)", str(p))
    return m.group(1) if m else None

hits["ref_accession"]   = hits["Ref_file"].apply(gtdb_acc)
hits["query_accession"] = hits["Query_file"].apply(query_acc)

print("\nTop-5 per query:")
for q, sub in hits.groupby("query_accession"):
    print(f"\n  query: {q}")
    cols = ["ref_accession", "ANI", "Align_fraction_query", "Align_fraction_ref"]
    print(sub[cols].sort_values("ANI", ascending=False).head(5).to_string(index=False))


hits.tsv: 1181 bytes

rows: 4, queries: 4

Top-5 per query:

  query: GCA_000008085.1
  ref_accession   ANI  Align_fraction_query  Align_fraction_ref
GCA_000008085.1 100.0                 100.0               100.0

  query: GCA_000010565.1
  ref_accession   ANI  Align_fraction_query  Align_fraction_ref
GCA_000010565.1 100.0                 100.0               100.0

  query: GCA_000145985.1
  ref_accession   ANI  Align_fraction_query  Align_fraction_ref
GCA_000145985.1 100.0                 100.0               100.0

  query: GCA_000147015.1
  ref_accession   ANI  Align_fraction_query  Align_fraction_ref
GCA_000147015.1 100.0                 100.0               100.0


## 5. Cross-check vs gtdbtk's closest_genome_reference

Pull the bac120 + ar53 summary TSVs from the cached gtdbtk demo run and confirm the top-1 skani
hit per query matches gtdbtk's reported `closest_genome_reference`. They should agree because
both calls go into the same R232 sketch bundle.


In [5]:
gtdbtk_summaries = [
    o.object_name for o in mincli.list_objects("cts", prefix=GTDBTK_DEMO_PREFIX, recursive=True)
    if (o.object_name.endswith("gtdbtk.bac120.summary.tsv")
        or o.object_name.endswith("gtdbtk.ar53.summary.tsv"))
    and "/classify/" not in o.object_name
    and "/identify/" not in o.object_name
    and "/align/"    not in o.object_name
    and "/ani_screen/" not in o.object_name
]
frames = []
for k in gtdbtk_summaries:
    body = mincli.get_object("cts", k).read().decode("utf-8")
    frames.append(pd.read_csv(io.StringIO(body), sep="\t", dtype=str))
gtdbtk_df = pd.concat(frames, ignore_index=True)
gtdbtk_df = gtdbtk_df[gtdbtk_df["classification"].notna()].copy()
gtdbtk_df["query_accession"] = gtdbtk_df["user_genome"].apply(query_acc)
gtdbtk_df["gtdbtk_ref_acc"]  = gtdbtk_df["closest_genome_reference"].apply(gtdb_acc)
gtdbtk_df["gtdbtk_ani"]      = pd.to_numeric(gtdbtk_df["closest_genome_ani"], errors="coerce")

skani_top = (hits.sort_values("ANI", ascending=False)
                 .drop_duplicates("query_accession", keep="first")
                 [["query_accession", "ref_accession", "ANI"]]
                 .rename(columns={"ref_accession": "skani_top1_ref", "ANI": "skani_top1_ani"}))

compare = gtdbtk_df[["query_accession", "gtdbtk_ref_acc", "gtdbtk_ani"]] \
    .merge(skani_top, on="query_accession", how="outer")
compare["agree"] = compare["gtdbtk_ref_acc"] == compare["skani_top1_ref"]
print(compare.to_string(index=False))


query_accession  gtdbtk_ref_acc  gtdbtk_ani  skani_top1_ref  skani_top1_ani  agree
GCA_000008085.1 GCA_000008085.1       100.0 GCA_000008085.1           100.0   True
GCA_000010565.1 GCA_000010565.1       100.0 GCA_000010565.1           100.0   True
GCA_000145985.1 GCA_000145985.1       100.0 GCA_000145985.1           100.0   True
GCA_000147015.1 GCA_000147015.1       100.0 GCA_000147015.1           100.0   True


## 6. End-to-end check

In [6]:
checks = {
    "job complete (exit 0)":          job.get_job_status()["state"] == "complete"
                                          and job.get_exit_codes().get("exit_codes") == [0],
    "hits.tsv written":                len(outs) == 1,
    "at least one hit per query":      hits["query_accession"].nunique() == len(input_files),
    "every query has a top-1 hit":     len(skani_top) == len(input_files),
    "top-1 skani vs gtdbtk agreement": compare["agree"].fillna(False).all(),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off; inspect above.")


  [x] job complete (exit 0)
  [x] hits.tsv written
  [x] at least one hit per query
  [x] every query has a top-1 hit
  [x] top-1 skani vs gtdbtk agreement

All green.
